# Scorecard de Lançamentos — Análise Unificada

Notebook unificado que combina:

1. **Scorecard quantitativo** — avaliação contínua com scores ponderados por pilar (JSON-driven)
2. **Checklist operacional** — avaliação binária (passa / não passa) por critério fixo
3. **Análises de negócio** — gráficos reportáveis para os times de Produto e Growth

### Fontes de dados
- **BigQuery** (`insider-data-lake`) — DRE, SKUs, estoque, pedidos de produção, reviews
- **Google Sheets** — exportação dos resultados para acompanhamento colaborativo

### Planilhas de exportação
| Planilha | Uso |
|---|---|
| [Scorecard](https://docs.google.com/spreadsheets/d/1aDvLGA1VwqR6a2wzdl1lOS0KepQuRKpIzklsORxxgkE) | Deepnote long + Deepnote short do scorecard |
| [Checklist](https://docs.google.com/spreadsheets/d/1aDvLGA1VwqR6a2wzdl1lOS0KepQuRKpIzklsORxxgkE/) | Output checklist com checkboxes |


## 0. Setup e Configurações

Todas as dependências, conexões e parâmetros manuais estão centralizados aqui.

- Edite `SCORECARD_INPUTS` para alterar benchmarks do scorecard.
- Edite `CHECKLIST_CRITERIA` para alterar os critérios binários do checklist.
- O restante do notebook é genérico e não precisar ser alterado.


In [1]:
from datetime import date
import json
from pathlib import Path
import re

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import gspread
from google.oauth2.service_account import Credentials
import google.auth

import pandas as pd
import numpy as np

# from insider_data_utils.bigquery_utils.dataframe_writter import load_df_to_bigquery
# from bigquery_utils import load_df_to_bigquery
service_account_info = os.getenv("BQ_SERVICE_ACCOUNT")

# Define the scope for the Google Sheets API
scope = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]

# Authentication - using a service account file
def get_service_account_credentials():
    import os
    import json

    # Path to the service account file
    return json.loads(os.getenv("BIGQUERY_INTEGRATION_SERVICE_ACCOUNT"))

# Extracting credentials
credentials_info = get_service_account_credentials()
cred_bundle = Credentials.from_service_account_info(credentials_info, scopes=scope)

# Authorize the client
client = gspread.authorize(cred_bundle)

In [2]:
# ── Planilhas de destino ──────────────────────────────────────────
SPREADSHEET_SCORECARD = client.open_by_url(
    # "https://docs.google.com/spreadsheets/d/185K3dq_QtsUZGjaDqL7redJxlbZP-G9JvXC5H-FNwoc/"
    "https://docs.google.com/spreadsheets/d/1aDvLGA1VwqR6a2wzdl1lOS0KepQuRKpIzklsORxxgkE"
)
SPREADSHEET_CHECKLIST = client.open_by_url(
    "https://docs.google.com/spreadsheets/d/1aDvLGA1VwqR6a2wzdl1lOS0KepQuRKpIzklsORxxgkE/"
)

# ── Inputs manuais do scorecard ───────────────────────────────────
# Edite aqui os benchmarks sem precisar alterar o JSON de config
SCORECARD_INPUTS = {
    "benchmark_mc3": 0.30,
    "benchmark_mrkup_entrada": 3.85,
}

# ── Critérios do checklist (avaliação binária) ────────────────────
# Cada critério: coluna da base, valor de referência e tipo de comparação
# Comparações suportadas: greater_equal, less_equal, equal
CHECKLIST_CRITERIA = {
    "Sellthrough": {
        "30 dias": {"column": "sellthrough_30d", "check_value": 0.35, "comparison": "greater_equal"},
        "60 dias": {"column": "sellthrough_60d", "check_value": 0.50, "comparison": "greater_equal"},
        "90 dias": {"column": "sellthrough_90d", "check_value": 0.65, "comparison": "greater_equal"},
    },
    "Unit Economics": {
        "Percentil Receita Categoria": {
            "column": "representatividade_categoria",
            "check_value": 0.5,
            "comparison": "greater_equal",
        },
        "MC3": {"column": "mc3", "check_value": 0.25, "comparison": "greater_equal"},
    },
    "Satisfação do cliente": {
        "Rating": {"column": "rating", "check_value": 4, "comparison": "greater_equal"},
        "Troca e Devolução": {"column": "tx_devolucao_categoria", "check_value": 0.058, "comparison": "less_equal"},
    },
}

# ── Path do JSON de configuração do scorecard ─────────────────────
SCORECARD_CONFIG_PATH = Path("JSON/scorecard_config_exemplo.json")

# ── Paleta de cores padrão para classificações ────────────────────
COLOR_MAP_CLASSIFICACAO = {"GO": "#2ecc71", "ITERAR": "#f39c12", "KILL": "#e74c3c"}


## 1. Carregar base de lançamentos

A query `SQL/base_unificada.sql` consolida dados de:
- **DRE** — receita, MC3, markup, devoluções
- **SKUs** — produtos em estado `ativo_em_lancamento`
- **Pedidos de produção** — quantidades planejadas e recebidas
- **Sell-through** — vendas líquidas em janelas de 30, 60 e 90 dias
- **Estoque** — posição atual de estoque virtual
- **Reviews** — rating médio e quantidade de avaliações no site

Esta base alimenta tanto o scorecard quanto o checklist.


In [3]:
if '_dntk' in globals():
  _dntk.dataframe_utils.configure_dataframe_formatter('{"sortBy":[],"filters":[],"conditionalFilters":[]}')
else:
  _deepnote_current_table_attrs = '{"sortBy":[],"filters":[],"conditionalFilters":[]}'

df_base = _dntk.execute_sql(
  'with dre_filtrado as (\n  select *\n  from `insider-data-lake.fpa.analytical_dre`\n  qualify sum(quantity) over(partition by date_trunc(order_date, month)) >= 100\n)\n\n, produtos_em_lancamento as (\n  select\n    product_name, \n    string_agg(distinct color, \', \') cores_lancamento\n  from `insider-data-lake.integrated.skus`\n  where sku_state = \'ativo_em_lancamento\'\n  group by product_name\n)\n\n, dados_dre_produto as (\n  select\n    s.product_name,\n    category_4,\n    min(date(order_date))                                           as data_lancamento,\n    sum(quantity)                                                   as quantidade_vendida,\n    sum(refunded_quantity)                                          as refunded_quantity,\n    safe_divide(sum(refunded_quantity), sum(quantity))              as tx_devolucao,\n    sum(revenue_after_taxes)                                        as receita_liquida,\n    avg(safe_divide(items_potential_revenue, quantity))             as full_price,\n    safe_divide(sum(sku_total_cost), sum(sold_quantity))            as cmv_med,\n    safe_divide(sum(revenue_after_discounts), sum(sku_total_cost))  as mrkup_med,\n    safe_divide(sum(revenue_after_refunds), sum(sku_total_cost))    as mrkup_entrada,\n    safe_divide(sum(net_profit_after_marketing_costs), sum(nullif(revenue_after_taxes, 0))) as mc3\n  from `insider-data-lake.fpa.analytical_dre`\n  left join `insider-data-lake.integrated.skus` s using(sku)\n  where sku_state in (\'ativo_perene\', \'ativo_capsula\', \'ativo_em_lancamento\', \'desativado\')\n  group by 1, 2\n)\n\n, representatividade_skp_categoria as (\n  select\n    product_name,\n    percent_rank() over(partition by category_4 order by revenue_after_discounts desc) as representatividade_categoria\n  from (\n    select\n      s.product_name,\n      category_4,\n      sum(revenue_after_discounts) as revenue_after_discounts\n    from `insider-data-lake.fpa.analytical_dre`\n    left join `insider-data-lake.integrated.skus` s using(sku)\n    where date(order_date) > \'2025-01-01\'\n    group by 1, 2\n  )\n)\n\n, tb_receita_media_por_mes as (\n  select\n    product_name,\n    category_4,\n    safe_divide(\n        sum(revenue_after_taxes),\n        count(distinct date_trunc(order_date, month))\n        ) as receita_med_mensal,\n    safe_divide(\n        sum(quantity),\n        count(distinct date_trunc(order_date, month))\n        ) as quantity_med_mensal,\n    avg(representatividade_categoria) as representatividade_categoria\n  from dre_filtrado\n  left join representatividade_skp_categoria using(product_name)\n  where date(order_date) > \'2025-01-01\'\n  group by 1, 2\n)\n\n, dados_dre_categoria as (\n  select\n    category_4,\n    avg(mc3) as mc3_categoria,\n    avg(receita_med_mensal) as receita_med_categoria,\n    safe_divide(sum(refunded_quantity), sum(quantidade_vendida)) as tx_devolucao_categoria\n  from dados_dre_produto\n  left join tb_receita_media_por_mes using(product_name, category_4)\n  group by 1\n)\n\n, tb_vendas_produtos as (\n  select\n    product_name,\n    sum(if(date(order_date) < data_lancamento + 30, non_refunded_quantity, null)) as vendas_30d_liquidas,\n    sum(if(date(order_date) < data_lancamento + 60, non_refunded_quantity, null)) as vendas_60d_liquidas,\n    sum(if(date(order_date) < data_lancamento + 90, non_refunded_quantity, null)) as vendas_90d_liquidas\n  from `insider-data-lake.fpa.analytical_dre`\n  left join dados_dre_produto using(product_name)\n  group by 1\n)\n\n, maximo_de_estoque_produto as (\n    select\n        product_name, \n        virtual_stock as max_quantity\n    from (\n        select\n            product_name, \n            stock_date, \n            sum(virtual_stock) virtual_stock\n\n        from integrated.stock\n        left join integrated.skus using(sku)\n        group by all\n    )\n    qualify row_number() over(partition by product_name order by virtual_stock desc) = 1\n)\n\n, dados_de_pedidos as (\n  select\n    product_name,\n    sum(planned_quantity) as planned_quantity_total,\n    sum(received_quantity) as received_quantity_total,\n    sum(if(dt_min_entry_warehouse < data_lancamento + 120, planned_quantity, 0)) as planned_quantity_120d,\n    sum(if(dt_min_entry_warehouse < data_lancamento + 120, received_quantity, 0)) as received_quantity_120d\n  from `insider-data-lake.sop_silver.supply_chain_efficiency_model_input`\n  left join dados_dre_produto using(product_name)\n  where current_production_stage not in (\'pending\', \'canceled\')\n  group by 1\n)\n, vol_planejado_por_produto as (\n    select\n        product_name,\n        planned_quantity_total,\n        received_quantity_total,\n        planned_quantity_120d,\n        received_quantity_120d, \n        CASE \n            WHEN product_name = \'Regata Nadador IN-ACTION Seamless Feminino\' THEN max_quantity\n            WHEN received_quantity_120d > 0 THEN received_quantity_120d\n            ELSE IF(planned_quantity_120d > 0, planned_quantity_120d, max_quantity)\n        END as vol_plan_st\n        \n    from dados_de_pedidos\n    left join maximo_de_estoque_produto using(product_name)\n)\n\n, estoque_atual as (\n  select\n    product_name,\n    sum(virtual_stock) as stock\n  from `insider-data-lake.integrated.stock`\n  left join `insider-data-lake.integrated.skus` using(sku)\n  where stock_date = current_date\n  group by 1\n)\n\n, product_name_correction as (\n  select\n    distinct o.data_source_product_id as product_id,\n    s.product_name\n  from `insider-data-lake.integrated.order_items` o\n  join `insider-data-lake.integrated.skus` s using(sku)\n  where o.product_title is not null\n    and s.sku_state in (\'ativo_em_lancamento\', \'ativo_perene\', \'ativo_capsula\')\n)\n\n, dados_review_no_site as (\n  select\n    product_name,\n    round(avg(rating), 2) as rating,\n    count(1) as qtd_reviews\n  from `insider-data-lake.integrated.product_reviews` pr\n  join product_name_correction pnc using(product_id)\n  group by 1\n  having qtd_reviews > 10\n)\n\nselect\n  product_name,\n  cores_lancamento,\n  category_4,\n  \'ativo_em_lancamento\' as sku_state,\n  data_lancamento,\n  date_diff(current_date, data_lancamento, day) as tempo_de_vendas,\n\n  planned_quantity_total,\n  received_quantity_total,\n  planned_quantity_120d,\n  received_quantity_120d,\n\n  quantidade_vendida,\n  refunded_quantity,\n\n  vendas_30d_liquidas,\n  vendas_60d_liquidas,\n  vendas_90d_liquidas,\n\n\n  safe_divide(vendas_30d_liquidas, vp.vol_plan_st) as sellthrough_30d,\n  \n  IF(\n    date_diff(current_date, data_lancamento, day) > 30, \n    safe_divide(vendas_60d_liquidas, vp.vol_plan_st), \n    null\n  ) as sellthrough_60d,\n  \n  IF(\n    date_diff(current_date, data_lancamento, day) > 60, \n    safe_divide(vendas_90d_liquidas, vp.vol_plan_st), \n    null\n  ) as sellthrough_90d,\n\n  receita_med_mensal,\n  receita_med_categoria,\n  receita_liquida,\n  receita_liquida as receita_liquida_scorecard,\n\n  representatividade_categoria,\n\n  rev.rating,\n  rev.qtd_reviews,\n\n\n  full_price,\n  cmv_med,\n  mrkup_med,\n  mrkup_entrada,\n  mc3,\n  mc3_categoria,\n\n  stock,\n  tx_devolucao,\n  tx_devolucao_categoria,\n  quantity_med_mensal,\n  cmv_med * stock as valor_em_estoque, \n\n  safe_divide((30 * stock), quantity_med_mensal) as cobertura_dias\n\nfrom dados_dre_produto skp\nleft join tb_receita_media_por_mes rmm using(product_name, category_4)\n-- left join dados_de_pedidos op using(product_name)\nleft join tb_vendas_produtos tvp using(product_name)\nleft join vol_planejado_por_produto vp using(product_name)\nleft join dados_dre_categoria cat4 using(category_4)\nleft join estoque_atual est using(product_name)\nleft join dados_review_no_site rev using(product_name)\ninner join produtos_em_lancamento using(product_name)',
  'SQL_5088268A_9640_458A_83D1_46B07B955FAE',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_base

,product_name,cores_lancamento,category_4,sku_state,data_lancamento,tempo_de_vendas,planned_quantity_total,received_quantity_total,planned_quantity_120d,received_quantity_120d,...,mrkup_med,mrkup_entrada,mc3,mc3_categoria,stock,tx_devolucao,tx_devolucao_categoria,quantity_med_mensal,valor_em_estoque,cobertura_dias
0,Top Sem Costura NuForm Feminino,"Comfy Brown, Preto, Smooth Beige",Woman Fitness/Sportswear Top Tops,ativo_em_lancamento,2026-08-10,10,2997,3118.0,2997,3118,...,3.501125,3.774195,0.368248,0.194035,2861,0.055357,0.070326,270.000000,134068.547124,317.888889
1,Saia Midi Tokyo FutureForm Feminino,"Marinho, New Khaki, Preto",Woman Casual Bottoms Skirts Midi,ativo_em_lancamento,2026-04-16,126,3598,5635.0,3598,5635,...,2.830441,3.215956,0.336215,0.349992,1195,0.088351,0.094277,182.000000,110257.059901,196.978022
2,Perfect Body Manga Longa Feminino,"Azul Marinho, Preto, Rust Brown",Woman Casual One Piece Body,ativo_em_lancamento,2025-09-15,339,6280,5297.0,4995,4886,...,2.761182,3.392687,0.216190,0.203131,254,0.088363,0.088504,464.083333,13483.346875,16.419465
3,Blusa Gola Alta Manga Longa Roll-IN Masculino,Preto,Man Casual Top T-Shirt,ativo_em_lancamento,2026-06-15,66,5905,1918.0,2035,1918,...,4.424322,4.653811,0.324188,-0.083471,264,0.073667,0.050945,605.666667,10706.871192,13.076500
4,Pijama SleepIN Masculino,Azul Marinho,Man Lifewear Pijamas Kit,ativo_em_lancamento,2026-08-08,12,4999,NaN,0,0,...,2.931004,3.243659,0.294526,0.294526,4241,0.023863,0.023863,18.000000,211129.586765,7068.333333
5,Camiseta Polo Grafeno Edition Masculino,"Dust Beige, Preto",Man Casual Top Polo Shirt,ativo_em_lancamento,2026-03-23,150,1040,860.0,1040,860,...,2.663452,3.096215,0.446131,0.404217,68,0.081227,0.059608,140.833333,9149.989650,14.485207
6,Perfect Body Decote U Feminino,"Azul Marinho, Preto, Rust Brown",Woman Casual One Piece Body,ativo_em_lancamento,2025-09-15,339,6474,5971.0,5994,5590,...,2.811464,3.460748,0.190073,0.203131,150,0.088627,0.088504,534.500000,7084.165151,8.419083
7,Camiseta Grafeno Edition Masculino,"Dust Beige, Preto",Man Casual Top T-Shirt,ativo_em_lancamento,2026-03-23,150,1874,1778.0,1874,1778,...,2.407114,2.782314,0.318333,-0.083471,276,0.089764,0.050945,287.166667,26578.762436,28.833430
8,Casaco de Tricot Future Knit Feminino,"Honey Mustard, Preto, Taupe",Woman Casual Top Jackets/Hoodies,ativo_em_lancamento,2026-05-31,81,2013,1328.0,1392,1328,...,2.661019,2.980877,0.477247,0.435234,782,0.105356,0.073605,186.500000,220215.474324,125.790885
9,Legging Fitness DopamIN Feminino,"Evergreen, Latte, Preto",Woman Fitness/Sportswear Bottoms Leggings/Pants,ativo_em_lancamento,2026-08-15,5,2157,2774.0,2157,2774,...,2.770518,3.303338,0.270341,0.252696,2562,0.069136,0.095524,31.000000,217198.822935,2479.354839


## 2. Scorecard — Avaliação Contínua

O scorecard atribui uma **nota de 0 a 100** para cada produto, ponderada por pilares:
- **Tração Comercial** — sell-through e receita relativa à categoria
- **Unit Economics** — MC3, markup de entrada, representatividade na categoria
- **Satisfação e Marca** — taxa de devolução relativa e rating no site

### Como funciona
- Cada critério avalia uma expressão sobre o `df_base` (definida no JSON de config)
- Regras de scoring mapeiam o valor calculado para pontos (0–100)
- Pesos de critério são normalizados dentro do pilar; pesos de pilar no total
- Quando `normalize_weights_on_available = True`, critérios sem dados não penalizam o produto

### Classificação
| Score | Classificação |
|---|---|
| ≥ 70 | **GO** — produto saudável, manter/escalar |
| ≥ 50 | **ITERAR** — análise necessária, ajustar |
| < 50 | **KILL** — candidato a descontinuação |


### 2.1 Carregar configuração do scorecard

O JSON de configuração (`scorecard_config_exemplo.json`) define para cada critério:
- `memoria_calculo` — descrição legível da fórmula
- `expression` — fórmula avaliada sobre o `df_base`
- `reference_columns` — colunas necessárias
- `scoring.rules` — regras ordenadas de pontuação

Os benchmarks (ex: MC3 alvo) são injetados via `SCORECARD_INPUTS` da célula de setup.


In [4]:
with open(SCORECARD_CONFIG_PATH, "r", encoding="utf-8") as f:
    scorecard_config = json.load(f)

print(f"Config carregada: {SCORECARD_CONFIG_PATH.resolve()}")
print(f"Pilares: {[p['name'] for p in scorecard_config['pillars']]}")
print(f"Classificações: {[r['label'] for r in scorecard_config['classification_rules']]}")


Config carregada: /datasets/_deepnote_work/JSON/scorecard_config_exemplo.json
Pilares: ['Tracao Comercial', 'Unit Economics', 'Satisfacao e Marca']
Classificações: ['GO', 'ITERAR', 'KILL']


In [5]:
def config_to_dataframe(config: dict, inputs: dict = None) -> pd.DataFrame:
    """Transforma a configuração do scorecard em tabela para revisão rápida."""
    rows = []
    for pillar in config["pillars"]:
        for criterion in pillar["criteria"]:
            rows.append({
                "pilar": pillar["name"],
                "peso_pilar": pillar.get("weight", 1),
                "criterio_id": criterion["id"],
                "criterio": criterion["name"],
                "peso_criterio": criterion.get("weight", 1),
                "memoria_calculo": criterion.get("memoria_calculo", ""),
                "expressao": criterion["calculation"]["expression"],
                "colunas_referencia": ", ".join(
                    criterion["calculation"].get("reference_columns", [])
                ),
                "regras_score": json.dumps(
                    criterion["scoring"].get("rules", []), ensure_ascii=False
                ),
                "default_points": criterion["scoring"].get("default_points"),
                "missing_points": criterion["scoring"].get("missing_points"),
            })
    return pd.DataFrame(rows)


df_config = config_to_dataframe(scorecard_config, SCORECARD_INPUTS)
display(df_config)


,pilar,peso_pilar,criterio_id,criterio,peso_criterio,memoria_calculo,expressao,colunas_referencia,regras_score,default_points,missing_points
0,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.30,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"[{""op"": "">="", ""value"": 0.4, ""points"": 100, ""la...",0,None
1,Tracao Comercial,0.2,sellthrough_60d,Sell-through 60 dias,0.30,Usa a coluna sellthrough_60d diretamente.,sellthrough_60d,sellthrough_60d,"[{""op"": "">="", ""value"": 0.65, ""points"": 100, ""l...",0,None
2,Tracao Comercial,0.2,sellthrough_90d,Sell-through 90 dias,0.25,Usa a coluna sellthrough_90d diretamente.,sellthrough_90d,sellthrough_90d,"[{""op"": "">="", ""value"": 0.85, ""points"": 100, ""l...",0,None
3,Tracao Comercial,0.2,receita_vs_categoria,Receita media mensal vs media da categoria,0.15,"safe_div(receita_med_mensal, receita_med_categ...","safe_div(receita_med_mensal, receita_med_categ...","receita_med_mensal, receita_med_categoria","[{""op"": "">="", ""value"": 1.2, ""points"": 100, ""la...",0,None
4,Unit Economics,0.4,mc3_vs_benchmark,MC3 vs benchmark manual,0.50,"safe_div(mc3, benchmark_mc3)","safe_div(mc3, benchmark_mc3)",mc3,"[{""op"": "">="", ""value"": 1.2, ""points"": 100, ""la...",0,None
5,Unit Economics,0.4,mrkup_entrada_vs_benchmark,Markup de entrada vs benchmark manual,0.30,"safe_div(mrkup_entrada, benchmark_mrkup_entrada)","safe_div(mrkup_entrada, benchmark_mrkup_entrada)",mrkup_entrada,"[{""op"": "">="", ""value"": 1.2, ""points"": 100, ""la...",0,None
6,Unit Economics,0.4,representatividade_categoria,Representatividade na categoria,0.20,Usa a coluna representatividade_categoria dire...,representatividade_categoria,representatividade_categoria,"[{""op"": "">="", ""value"": 0.8, ""points"": 100, ""la...",0,None
7,Satisfacao e Marca,0.4,devolucao_relativa_categoria,Taxa de devolucao relativa a categoria,0.60,"safe_div(tx_devolucao, tx_devolucao_categoria)","safe_div(tx_devolucao, tx_devolucao_categoria)","tx_devolucao, tx_devolucao_categoria","[{""op"": ""<="", ""value"": 0.8, ""points"": 100, ""la...",0,None
8,Satisfacao e Marca,0.4,rating,Avaliacao media,0.40,Usa a coluna rating diretamente.,rating,rating,"[{""op"": "">="", ""value"": 4.6, ""points"": 100, ""la...",0,None


### 2.2 Motor de Scorecard

Funções do motor:
- `safe_div` — divisão segura evitando zero / infinito
- `validate_config` — valida JSON e checa colunas de referência no df
- `evaluate_expression` — avalia fórmulas sobre as colunas do df + inputs
- `apply_scoring_rules` — aplica faixas de pontuação na ordem do JSON
- `resolve_null_scores` — política de nulos (`redistribute` ou `penalize`)
- `weighted_average_from_columns` — média ponderada com normalização opcional
- `classify_scores` — classifica o score total em GO / ITERAR / KILL
- `apply_scorecard` — orquestra tudo e retorna `df_scorecard` + `df_memorial`


In [6]:
# ── Funções auxiliares do motor ───────────────────────────────────

def slugify(text: str) -> str:
    """Converte texto em slug seguro para nomes de coluna."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def ensure_series(obj, index: pd.Index) -> pd.Series:
    """Converte escalares ou arrays em Series com o mesmo index do df."""
    if isinstance(obj, pd.Series):
        return obj.reindex(index)
    if np.isscalar(obj):
        return pd.Series(obj, index=index)
    return pd.Series(obj, index=index)


def safe_div(numerador, denominador):
    """Divisão segura para Series, com tratamento de zero e infinito."""
    idx = numerador.index if isinstance(numerador, pd.Series) else None
    if idx is None and isinstance(denominador, pd.Series):
        idx = denominador.index
    if idx is None:
        raise ValueError("safe_div precisa de pelo menos um argumento em formato Series.")

    num = pd.to_numeric(ensure_series(numerador, idx), errors="coerce")
    den = pd.to_numeric(ensure_series(denominador, idx), errors="coerce").replace(0, np.nan)
    out = num / den
    return out.replace([np.inf, -np.inf], np.nan)


# ── Validação ─────────────────────────────────────────────────────

def validate_config(df: pd.DataFrame, config: dict) -> None:
    """Valida a estrutura mínima do JSON e as colunas de referência."""
    if "pillars" not in config:
        raise ValueError("A configuração precisa da chave 'pillars'.")

    valid_null_handling = {"redistribute", "penalize"}

    missing_columns = []
    for pillar in config["pillars"]:
        if "criteria" not in pillar:
            raise ValueError(f"O pilar {pillar.get('name')} precisa da chave 'criteria'.")

        null_score = pillar.get("all_criteria_null_score")
        if null_score is not None and not isinstance(null_score, (int, float)):
            raise ValueError(
                f"O pilar {pillar.get('name')} tem all_criteria_null_score inválido."
            )

        for criterion in pillar["criteria"]:
            calc = criterion.get("calculation", {})
            expr = calc.get("expression")
            if not expr:
                raise ValueError(
                    f"O critério {criterion.get('id')} precisa de 'calculation.expression'."
                )

            handling = criterion.get("null_value_handling", "redistribute")
            if handling not in valid_null_handling:
                raise ValueError(
                    f"O critério {criterion.get('id')} tem null_value_handling='{handling}' "
                    f"inválido. Valores aceitos: {valid_null_handling}"
                )

            for col in calc.get("reference_columns", []):
                if col not in df.columns:
                    missing_columns.append((criterion.get("id"), col))

            if "scoring" not in criterion:
                raise ValueError(f"O critério {criterion.get('id')} precisa de 'scoring'.")

    if missing_columns:
        msg = "\n".join(
            [f"- critério '{crit}': coluna ausente '{col}'" for crit, col in missing_columns]
        )
        raise KeyError("Colunas faltantes no df_base:\n" + msg)


# ── Avaliação de expressões ───────────────────────────────────────

def evaluate_expression(df: pd.DataFrame, criterion: dict, inputs: dict = None) -> pd.Series:
    """Avalia a fórmula do critério usando colunas do df e inputs externos."""
    calc = criterion["calculation"]
    expr = calc["expression"]

    local_env = {col: df[col] for col in df.columns}
    if inputs:
        local_env.update(inputs)
    local_env.update({"np": np, "pd": pd, "safe_div": safe_div})

    result = eval(expr, {"__builtins__": {}}, local_env)
    result = ensure_series(result, df.index)
    result = pd.to_numeric(result, errors="coerce")
    return result.replace([np.inf, -np.inf], np.nan)


# ── Scoring ───────────────────────────────────────────────────────

def compare_series(values: pd.Series, op: str, target) -> pd.Series:
    """Aplica um comparador entre a série e um alvo."""
    ops = {
        ">=": lambda v, t: v >= t,
        ">": lambda v, t: v > t,
        "<=": lambda v, t: v <= t,
        "<": lambda v, t: v < t,
        "==": lambda v, t: v == t,
        "!=": lambda v, t: v != t,
    }
    if op == "between":
        lower, upper = target
        return values.between(lower, upper, inclusive="both")
    if op in ops:
        return ops[op](values, target)
    raise ValueError(f"Operador não suportado: {op}")


def apply_scoring_rules(values: pd.Series, scoring: dict):
    """Aplica as faixas de score na ordem em que aparecem no JSON."""
    default_points = scoring.get("default_points", 0)
    missing_points = scoring.get("missing_points", np.nan)
    default_label = scoring.get("default_label", "default")

    scores = pd.Series(np.nan, index=values.index, dtype="float64")
    labels = pd.Series(pd.NA, index=values.index, dtype="object")

    missing_mask = values.isna()
    scores.loc[missing_mask] = missing_points
    labels.loc[missing_mask] = "missing"

    remaining = values.notna().copy()
    for i, rule in enumerate(scoring.get("rules", []), start=1):
        op = rule["op"]
        target = rule["value"]
        mask = remaining & compare_series(values, op, target)
        if mask.any():
            scores.loc[mask] = rule["points"]
            labels.loc[mask] = rule.get("label", f"regra_{i}")
            remaining.loc[mask] = False

    scores.loc[remaining] = default_points
    labels.loc[remaining] = default_label
    return scores, labels


def resolve_null_scores(
    values: pd.Series, scores: pd.Series, labels: pd.Series, handling: str
) -> tuple:
    """Aplica a política de tratamento de nulos do critério.

    - 'redistribute': força score=NaN → peso redistribuído aos demais
    - 'penalize': mantém missing_points como score
    """
    if handling == "redistribute":
        null_mask = values.isna()
        scores = scores.copy()
        labels = labels.copy()
        scores.loc[null_mask] = np.nan
        labels.loc[null_mask] = "redistributed"
    return scores, labels


# ── Média ponderada ───────────────────────────────────────────────

def weighted_average_from_columns(
    df: pd.DataFrame,
    col_weights: dict,
    normalize_available: bool = True,
) -> pd.Series:
    """Calcula média ponderada a partir de colunas já calculadas no df."""
    if not col_weights:
        return pd.Series(np.nan, index=df.index)

    weights = pd.Series(col_weights, dtype="float64")
    weights = weights / weights.sum()

    if normalize_available:
        weighted_sum = pd.Series(0.0, index=df.index)
        weight_sum = pd.Series(0.0, index=df.index)

        for col, weight in weights.items():
            valid = df[col].notna().astype(float)
            weighted_sum = weighted_sum + df[col].fillna(0) * weight
            weight_sum = weight_sum + valid * weight

        return weighted_sum / weight_sum.replace(0, np.nan)

    out = pd.Series(0.0, index=df.index)
    for col, weight in weights.items():
        out = out + df[col].fillna(0) * weight
    return out


# ── Classificação ─────────────────────────────────────────────────

def classify_scores(score_series: pd.Series, classification_rules: list) -> pd.Series:
    """Classifica o score total conforme as regras de corte."""
    rules = sorted(classification_rules, key=lambda x: x["min_score"], reverse=True)
    labels = pd.Series(pd.NA, index=score_series.index, dtype="object")

    remaining = score_series.notna().copy()
    for rule in rules:
        mask = remaining & (score_series >= rule["min_score"])
        labels.loc[mask] = rule["label"]
        remaining.loc[mask] = False

    return labels


# ── Orquestrador principal ────────────────────────────────────────

def apply_scorecard(df: pd.DataFrame, config: dict, inputs: dict = None):
    """Aplica o scorecard ao df_base e devolve (df_resultado, df_memorial)."""
    validate_config(df, config)

    result = df.copy()
    memorial_frames = []

    normalize_available = config.get("options", {}).get("normalize_weights_on_available", True)
    id_columns = [c for c in config.get("id_columns", []) if c in result.columns]
    if not id_columns:
        id_columns = [
            c
            for c in ["product_name", "category_4", "sku_state", "data_lancamento"]
            if c in result.columns
        ]

    pillar_score_col_weights = {}
    for pillar in config["pillars"]:
        pillar_name = pillar["name"]
        pillar_slug = slugify(pillar_name)
        pillar_weight = pillar.get("weight", 1)
        criteria = pillar.get("criteria", [])

        crit_weights = {
            criterion["id"]: float(criterion.get("weight", 1)) for criterion in criteria
        }
        crit_weight_total = sum(crit_weights.values()) or 1.0
        crit_weights = {k: v / crit_weight_total for k, v in crit_weights.items()}

        score_col_weights = {}
        for criterion in criteria:
            crit_id = criterion["id"]
            crit_slug = slugify(crit_id)
            handling = criterion.get("null_value_handling", "redistribute")

            metric_col = f"valor__{pillar_slug}__{crit_slug}"
            score_col = f"score__{pillar_slug}__{crit_slug}"
            faixa_col = f"faixa__{pillar_slug}__{crit_slug}"

            values = evaluate_expression(result, criterion, inputs)
            scores, labels = apply_scoring_rules(values, criterion["scoring"])
            scores, labels = resolve_null_scores(values, scores, labels, handling)

            result[metric_col] = values
            result[score_col] = scores
            result[faixa_col] = labels
            score_col_weights[score_col] = crit_weights[crit_id]

            memorial = result[id_columns].copy()
            memorial["pilar"] = pillar_name
            memorial["peso_pilar"] = pillar_weight
            memorial["criterio_id"] = crit_id
            memorial["criterio"] = criterion["name"]
            memorial["peso_criterio_no_pilar"] = crit_weights[crit_id]
            memorial["null_value_handling"] = handling
            memorial["memoria_calculo"] = criterion.get("memoria_calculo", "")
            memorial["expressao"] = criterion["calculation"]["expression"]
            memorial["colunas_referencia"] = ", ".join(
                criterion["calculation"].get("reference_columns", [])
            )
            memorial["inputs"] = json.dumps(inputs or {}, ensure_ascii=False)
            memorial["valor_calculado"] = values.values
            memorial["score_criterio"] = scores.values
            memorial["faixa_aplicada"] = labels.values
            memorial_frames.append(memorial)

        pillar_score_col = f"score_pilar__{pillar_slug}"
        result[pillar_score_col] = weighted_average_from_columns(
            result, score_col_weights, normalize_available=normalize_available,
        )

        # Quando todos os critérios do pilar são nulos, usa score fixo
        all_null_score = pillar.get("all_criteria_null_score")
        if all_null_score is not None:
            result[pillar_score_col] = result[pillar_score_col].fillna(all_null_score)

        pillar_score_col_weights[pillar_score_col] = pillar_weight

    result["score_total"] = weighted_average_from_columns(
        result, pillar_score_col_weights, normalize_available=normalize_available,
    )
    result["classificacao"] = classify_scores(
        result["score_total"], config["classification_rules"]
    )

    df_memorial = pd.concat(memorial_frames, axis=0, ignore_index=True)
    return result, df_memorial


### 2.3 Aplicar o scorecard

A saída principal é `df_scorecard` com métricas calculadas, score por critério,
score por pilar, score total e classificação GO / ITERAR / KILL.

O `df_memorial` é um registro em formato longo que facilita auditoria e exportação.


In [7]:
df_scorecard, df_memorial = apply_scorecard(df_base, scorecard_config, SCORECARD_INPUTS)

# Colunas de preview
score_cols_preview = ["classificacao"] + [
    c for c in df_scorecard.columns if c.startswith("score_pilar__")
] + ["score_total"]

id_cols_preview = [
    c for c in ["product_name", "category_4", "data_lancamento", "tempo_de_vendas"]
    if c in df_scorecard.columns
]

df_scorecard[id_cols_preview + score_cols_preview]


,product_name,category_4,data_lancamento,tempo_de_vendas,classificacao,score_pilar__tracao_comercial,score_pilar__unit_economics,score_pilar__satisfacao_e_marca,score_total
0,Top Sem Costura NuForm Feminino,Woman Fitness/Sportswear Top Tops,2026-08-10,10,GO,33.333333,80.0,100.0,78.666667
1,Saia Midi Tokyo FutureForm Feminino,Woman Casual Bottoms Skirts Midi,2026-04-16,126,ITERAR,0.000000,74.0,88.0,64.800000
2,Perfect Body Manga Longa Feminino,Woman Casual One Piece Body,2025-09-15,339,ITERAR,34.500000,53.0,88.0,63.300000
3,Blusa Gola Alta Manga Longa Roll-IN Masculino,Man Casual Top T-Shirt,2026-06-15,66,ITERAR,80.000000,76.0,32.0,59.200000
4,Pijama SleepIN Masculino,Man Lifewear Pijamas Kit,2026-08-08,12,ITERAR,26.666667,48.0,80.0,56.533333
5,Camiseta Polo Grafeno Edition Masculino,Man Casual Top Polo Shirt,2026-03-23,150,ITERAR,74.000000,74.0,30.0,56.400000
6,Perfect Body Decote U Feminino,Woman Casual One Piece Body,2025-09-15,339,ITERAR,68.000000,33.0,76.0,57.200000
7,Camiseta Grafeno Edition Masculino,Man Casual Top T-Shirt,2026-03-23,150,KILL,51.000000,55.0,24.0,41.800000
8,Casaco de Tricot Future Knit Feminino,Woman Casual Top Jackets/Hoodies,2026-05-31,81,KILL,43.500000,71.0,0.0,37.100000
9,Legging Fitness DopamIN Feminino,Woman Fitness/Sportswear Bottoms Leggings/Pants,2026-08-15,5,ITERAR,0.000000,68.0,100.0,67.200000


In [8]:
# Memorial de cálculo em formato longo (útil para auditoria)
print(f"Memorial: {len(df_memorial)} registros ({len(df_scorecard)} produtos × critérios)")
display(df_memorial.head(20))


Memorial: 216 registros (24 produtos × critérios)


,product_name,category_4,sku_state,data_lancamento,pilar,peso_pilar,criterio_id,criterio,peso_criterio_no_pilar,null_value_handling,memoria_calculo,expressao,colunas_referencia,inputs,valor_calculado,score_criterio,faixa_aplicada
0,Top Sem Costura NuForm Feminino,Woman Fitness/Sportswear Top Tops,ativo_em_lancamento,2026-08-10,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.076204,0.0,default
1,Saia Midi Tokyo FutureForm Feminino,Woman Casual Bottoms Skirts Midi,ativo_em_lancamento,2026-04-16,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.039042,0.0,default
2,Perfect Body Manga Longa Feminino,Woman Casual One Piece Body,ativo_em_lancamento,2025-09-15,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.169873,30.0,fraco
3,Blusa Gola Alta Manga Longa Roll-IN Masculino,Man Casual Top T-Shirt,ativo_em_lancamento,2026-06-15,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.547445,100.0,muito forte
4,Pijama SleepIN Masculino,Man Lifewear Pijamas Kit,ativo_em_lancamento,2026-08-08,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.004137,0.0,default
5,Camiseta Polo Grafeno Edition Masculino,Man Casual Top Polo Shirt,ativo_em_lancamento,2026-03-23,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.422093,100.0,muito forte
6,Perfect Body Decote U Feminino,Woman Casual One Piece Body,ativo_em_lancamento,2025-09-15,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.264580,60.0,ok
7,Camiseta Grafeno Edition Masculino,Man Casual Top T-Shirt,ativo_em_lancamento,2026-03-23,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.253093,60.0,ok
8,Casaco de Tricot Future Knit Feminino,Woman Casual Top Jackets/Hoodies,ativo_em_lancamento,2026-05-31,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.327560,60.0,ok
9,Legging Fitness DopamIN Feminino,Woman Fitness/Sportswear Bottoms Leggings/Pants,ativo_em_lancamento,2026-08-15,Tracao Comercial,0.2,sellthrough_30d,Sell-through 30 dias,0.3,redistribute,Usa a coluna sellthrough_30d diretamente.,sellthrough_30d,sellthrough_30d,"{""benchmark_mc3"": 0.3, ""benchmark_mrkup_entrad...",0.009377,0.0,default


In [9]:
# ── Resumo por classificação ─────────────────────────────────────
if "classificacao" in df_scorecard.columns:
    resumo_classificacao = (
        df_scorecard["classificacao"]
        .value_counts(dropna=False)
        .rename_axis("classificacao")
        .reset_index(name="qtd_produtos")
    )
    display(resumo_classificacao)

# ── Médias de score por pilar ────────────────────────────────────
pillar_cols = [c for c in df_scorecard.columns if c.startswith("score_pilar__")]
if pillar_cols:
    medias_pilares = (
        df_scorecard[pillar_cols]
        .mean()
        .sort_values(ascending=False)
        .rename("media_score")
        .reset_index()
        .rename(columns={"index": "pilar"})
    )
    display(medias_pilares)


,classificacao,qtd_produtos
0,ITERAR,11
1,GO,7
2,KILL,6


,pilar,media_score
0,score_pilar__unit_economics,70.875000
1,score_pilar__satisfacao_e_marca,60.750000
2,score_pilar__tracao_comercial,29.256944


## 3. Checklist Operacional — Avaliação Binária

O checklist avalia cada produto em critérios do tipo **passa / não passa** (booleano).
Ao contrário do scorecard (que dá nota contínua), aqui cada critério tem um threshold fixo.

Os critérios são definidos em `CHECKLIST_CRITERIA` na célula de setup.

> **Uso típico:** Validação rápida em reuniões de portfólio — o produto atende os mínimos?


In [10]:
# ── 4.1 Distribuição de Classificação (GO / ITERAR / KILL) ────────
# Visão executiva: quantos lançamentos estão saudáveis vs em risco?

fig = px.pie(
    df_scorecard,
    names="classificacao",
    title="Distribuição dos Lançamentos por Classificação",
    color="classificacao",
    color_discrete_map=COLOR_MAP_CLASSIFICACAO,
    hole=0.4,
)
fig.update_traces(textinfo="label+percent+value")
fig.update_layout(width=700, height=450)
fig.show()


In [11]:
# ── Comparadores suportados ───────────────────────────────────────
_COMPARATORS = {
    "greater_equal": lambda val, ref: val >= ref,
    "less_equal": lambda val, ref: val <= ref,
    "equal": lambda val, ref: val == ref,
}


def apply_checklist(df: pd.DataFrame, criteria: dict) -> pd.DataFrame:
    """Avalia os critérios binários do checklist para cada produto.

    Retorna um DataFrame com uma coluna booleana e uma coluna de valor
    para cada subcritério.
    """
    records = []
    for produto in df["product_name"].unique():
        row = df[df["product_name"] == produto].iloc[0]
        result = {"Produto": produto}
        for grupo, subcrit in criteria.items():
            for nome, spec in subcrit.items():
                key = f"{grupo} - {nome}"
                comparator = _COMPARATORS[spec["comparison"]]
                result[key] = bool(comparator(row[spec["column"]], spec["check_value"]))
                result[f"{key} value"] = row[spec["column"]]
        records.append(result)
    return pd.DataFrame(records)


df_checklist = apply_checklist(df_base, CHECKLIST_CRITERIA)
print(f"Checklist: {len(df_checklist)} produtos avaliados")
df_checklist


Checklist: 24 produtos avaliados


,Produto,Sellthrough - 30 dias,Sellthrough - 30 dias value,Sellthrough - 60 dias,Sellthrough - 60 dias value,Sellthrough - 90 dias,Sellthrough - 90 dias value,Unit Economics - Percentil Receita Categoria,Unit Economics - Percentil Receita Categoria value,Unit Economics - MC3,Unit Economics - MC3 value,Satisfação do cliente - Rating,Satisfação do cliente - Rating value,Satisfação do cliente - Troca e Devolução,Satisfação do cliente - Troca e Devolução value
0,Top Sem Costura NuForm Feminino,False,0.076204,False,NaN,False,NaN,False,0.444444,True,0.368248,False,NaN,False,0.070326
1,Saia Midi Tokyo FutureForm Feminino,False,0.039042,False,0.083052,False,0.116770,True,0.666667,True,0.336215,True,4.86,False,0.094277
2,Perfect Body Manga Longa Feminino,False,0.169873,False,0.345682,False,0.515759,True,1.000000,False,0.216190,True,4.94,False,0.088504
3,Blusa Gola Alta Manga Longa Roll-IN Masculino,True,0.547445,True,0.805621,True,0.839908,False,0.354167,True,0.324188,True,4.24,True,0.050945
4,Pijama SleepIN Masculino,False,0.004137,False,NaN,False,NaN,False,0.000000,True,0.294526,False,NaN,True,0.023863
5,Camiseta Polo Grafeno Edition Masculino,True,0.422093,True,0.629070,True,0.789535,False,0.333333,True,0.446131,False,NaN,False,0.059608
6,Perfect Body Decote U Feminino,False,0.264580,False,0.487835,True,0.729338,False,0.000000,False,0.190073,True,4.60,False,0.088504
7,Camiseta Grafeno Edition Masculino,False,0.253093,False,0.393701,False,0.600675,False,0.312500,True,0.318333,True,4.13,True,0.050945
8,Casaco de Tricot Future Knit Feminino,False,0.327560,False,0.422297,False,0.488359,True,0.500000,True,0.477247,False,NaN,False,0.073605
9,Legging Fitness DopamIN Feminino,False,0.009377,False,NaN,False,NaN,True,0.875000,True,0.270341,False,NaN,False,0.095524


## 4. Análises de Negócio

Gráficos voltados para **tomada de decisão** por dois públicos:

| Time | Acionáveis |
|---|---|
| **Produto (físico)** | Identificar problemas de qualidade (devoluções, ratings baixos), priorizar melhorias |
| **Growth** | Identificar oportunidades de venda (sell-through, receita), priorizar investimento em mídia |

Cada gráfico acompanha uma breve interpretação para facilitar a leitura em reuniões.


In [12]:
# ── 4.2 Score Médio por Categoria ─────────────────────────────────
# Acionável: quais categorias consistentemente lançam bem (ou mal)?
# Orienta decisões de portfólio e investimento em P&D por categoria.

df_cat = (
    df_scorecard
    .loc[~df_scorecard['product_name'].str.contains('Ziraldo')]
    .groupby("category_4")["score_total"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "score_medio", "count": "qtd_produtos"})
    .sort_values("score_medio", ascending=True)
)

fig = px.bar(
    df_cat,
    x="score_medio",
    y="category_4",
    orientation="h",
    text="qtd_produtos",
    color="score_medio",
    color_continuous_scale=["#e74c3c", "#f39c12", "#2ecc71"],
    title="Score Médio por Categoria — Quais categorias lançam melhor?",
    labels={"score_medio": "Score Médio", "category_4": "Categoria", "qtd_produtos": "Qtd"},
)
fig.update_traces(texttemplate="n=%{text}", textposition="inside")
fig.update_layout(width=900, height=max(400, len(df_cat) * 40))
fig.show()


In [13]:
# ── 4.3 Quadrante: Tração Comercial × Unit Economics ─────────────
# Leitura por quadrante:
#   ↗ Alto-Alto  = produto estrela (GO)
#   ↖ Alto-Baixo = vende bem mas não dá lucro → rever pricing / CMV
#   ↘ Baixo-Alto = lucrativo mas não vende → oportunidade para Growth
#   ↙ Baixo-Baixo = candidato a KILL

pillar_tracao = [
    c for c in df_scorecard.columns
    if "tracao_comercial" in c and c.startswith("score_pilar__")
]
pillar_unit = [
    c for c in df_scorecard.columns
    if "unit_economics" in c and c.startswith("score_pilar__")
]

if pillar_tracao and pillar_unit:
    col_tracao = pillar_tracao[0]
    col_unit = pillar_unit[0]

    df_quad = (
        df_scorecard
        .loc[~df_scorecard['product_name'].str.contains('Ziraldo')]
        .dropna(subset=[col_tracao, col_unit])
        .copy()
        )
    
    df_quad["receita_med_mensal"] = df_quad["receita_med_mensal"].clip(lower=0)


    fig = px.scatter(
        df_quad,
        x=col_tracao,
        y=col_unit,
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        hover_name="product_name",
        size="receita_med_mensal",
        title="Quadrante: Tração Comercial × Unit Economics",
        labels={
            col_tracao: "Tração Comercial (score)",
            col_unit: "Unit Economics (score)",
        },
    )
    fig.add_hline(y=50, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=50, line_dash="dash", line_color="gray", opacity=0.5)
    fig.update_layout(width=900, height=600)
    fig.show()
else:
    print("Colunas de pilar não encontradas — verifique o JSON de config.")


In [14]:
# ── 4.4 Evolução do Sell-through (30d → 60d → 90d) ───────────────
# Acionável para Growth: produtos que desaceleraram precisam de push;
# produtos que aceleraram podem ser escalados.

cols_st = ["sellthrough_30d", "sellthrough_60d", "sellthrough_90d"]
if all(c in df_scorecard.columns for c in cols_st):
    df_st = (
        df_scorecard[["product_name", "classificacao"] + cols_st]
        .loc[~df_scorecard['product_name'].str.contains('Ziraldo')]
        .melt(
            id_vars=["product_name", "classificacao"],
            value_vars=cols_st,
            var_name="janela",
            value_name="sellthrough",
        )
    )
    df_st["janela"] = df_st["janela"].map({
        "sellthrough_30d": "30 dias",
        "sellthrough_60d": "60 dias",
        "sellthrough_90d": "90 dias",
    })

    df_st['printing_name'] = df_st['product_name'].apply(lambda x: ' '.join(str(x).split(' ')[:4]))

    fig = px.bar(
        df_st.sort_values(["printing_name", "janela"]),
        x="printing_name",
        y="sellthrough",
        color="janela",
        barmode="group",
        title="Sell-through Progressivo por Produto — Evolução da Tração",
        labels={
            "sellthrough": "Sell-through",
            "printing_name": "Produto",
            "janela": "Janela",
        },
        color_discrete_sequence=["#3498db", "#2980b9", "#1a5276"],
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        width=max(900, len(df_scorecard) * 80),
        height=500,
        yaxis_tickformat=".0%",
    )
    fig.show()


In [15]:
# ── 4.5 Devolução Relativa à Categoria — Alerta de Qualidade ─────
# Acionável para Produto: produtos com barra ACIMA da linha vermelha
# estão devolvendo mais que a média da categoria → investigar feedback,
# sizing, material, etc.

if "tx_devolucao" in df_scorecard.columns and "tx_devolucao_categoria" in df_scorecard.columns:
    df_dev = (
        df_scorecard[["product_name", "category_4", "tx_devolucao", "tx_devolucao_categoria"]]
        .loc[~df_scorecard['product_name'].str.contains('Ziraldo')]
        .dropna()
        .copy()
    )
    df_dev["ratio_devolucao"] = df_dev["tx_devolucao"] / df_dev["tx_devolucao_categoria"]
    df_dev = df_dev.sort_values("ratio_devolucao", ascending=False)

    df_dev['printing_name'] = df_dev['product_name'].apply(lambda x: ' '.join(str(x).split(' ')[:4]))


    fig = px.bar(
        df_dev,
        x="printing_name",
        y="ratio_devolucao",
        color="ratio_devolucao",
        color_continuous_scale=["#2ecc71", "#f39c12", "#e74c3c"],
        title="Devolução Relativa à Categoria — Alerta para Time de Produto",
        labels={
            "ratio_devolucao": "Tx Devolução / Tx Categoria",
            "printing_name": "Produto",
        },
        hover_data=["category_4", "tx_devolucao", "tx_devolucao_categoria"],
    )
    fig.add_hline(
        y=1.0, line_dash="dash", line_color="red",
        annotation_text="Média da Categoria",
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        width=max(900, len(df_dev) * 70),
        height=500,
    )
    fig.show()


In [16]:
# ── 4.6 Risco de Estoque: Score × Valor em Estoque ───────────────
# Acionável: produtos NO CANTO INFERIOR DIREITO (score baixo + estoque alto)
# representam capital parado em risco — prioridade para liquidação ou markdown.

if "valor_em_estoque" in df_scorecard.columns:
    df_risk = df_scorecard.dropna(subset=["score_total", "valor_em_estoque"]).copy()

    df_risk['stock_size'] = df_risk['stock'].clip(lower=0)**0.3

    fig = px.scatter(
        df_risk,
        x="score_total",
        y="valor_em_estoque",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        hover_name="product_name",
        size="stock_size",
        title="Risco de Estoque: Score × Valor em Estoque (R$)",
        labels={
            "score_total": "Score Total",
            "valor_em_estoque": "Valor em Estoque (R$)",
        },
    )
    fig.add_vline(
        x=50, line_dash="dash", line_color="red", opacity=0.5,
        annotation_text="Limiar KILL/ITERAR",
    )
    fig.add_vline(
        x=70, line_dash="dash", line_color="green", opacity=0.5,
        annotation_text="Limiar ITERAR/GO",
    )
    fig.update_layout(width=900, height=600)
    fig.show()

# fig2 = px.scatter(
#     df_risk,
#     x="score_total",
#     y="valor_em_estoque",
#     color="classificacao",
#     color_discrete_map=COLOR_MAP_CLASSIFICACAO,
#     hover_name="product_name",
#     size="stock_size",
#     title="Risco de Estoque: Score × Valor em Estoque (R$)",
#     labels={
#         "score_total": "Score Total",
#         "valor_em_estoque": "Valor em Estoque (R$)",
#     },
# )
# fig2.add_vline(
#     x=50, line_dash="dash", line_color="red", opacity=0.5,
#     annotation_text="Limiar KILL/ITERAR",
# )
# fig2.add_vline(
#     x=70, line_dash="dash", line_color="green", opacity=0.5,
#     annotation_text="Limiar ITERAR/GO",
# )
# fig2.update_layout(width=900, height=600)
# fig2.show()


In [17]:
# ── 4.7 Rating dos Produtos no Site ──────────────────────────────
# Acionável para Produto: ratings abaixo de 4.0 merecem investigação
# de reviews qualitativos (sizing, conforto, durabilidade).

if "rating" in df_scorecard.columns:
    df_rating = (
        df_scorecard[["product_name", "rating", "qtd_reviews"]]
        .dropna()
        .sort_values("rating")
    )

    if not df_rating.empty:
        fig = px.bar(
            df_rating,
            x="rating",
            y="product_name",
            orientation="h",
            text="qtd_reviews",
            color="rating",
            color_continuous_scale=["#e74c3c", "#f39c12", "#2ecc71"],
            title="Rating dos Lançamentos no Site — Satisfação do Cliente",
            labels={
                "rating": "Rating Médio",
                "product_name": "Produto",
                "qtd_reviews": "Reviews",
            },
        )
        fig.update_traces(texttemplate="%{text} reviews", textposition="outside")
        fig.add_vline(
            x=4.5, line_dash="dash", line_color="gray",
            annotation_text="Meta 4.5",
        )
        fig.update_layout(width=900, height=max(400, len(df_rating) * 35))
        fig.show()
    else:
        print("Nenhum produto com dados de rating disponíveis.")


In [18]:
# ── 4.8 Ranking Final de Produtos por Score Total ────────────────
# Visão executiva: lista ordenada de todos os lançamentos com suas
# classificações, facilitando a priorização em reuniões de portfólio.

df_ranking = (
    df_scorecard[["product_name", "classificacao", "score_total"]]
    .dropna()
    .sort_values("score_total")
)

fig = px.bar(
    df_ranking,
    x="score_total",
    y="product_name",
    orientation="h",
    color="classificacao",
    color_discrete_map=COLOR_MAP_CLASSIFICACAO,
    title="Ranking de Lançamentos por Score Total",
    labels={"score_total": "Score Total", "product_name": "Produto"},
)
fig.add_vline(x=70, line_dash="dot", line_color="#2ecc71", annotation_text="GO ≥70")
fig.add_vline(x=50, line_dash="dot", line_color="#f39c12", annotation_text="ITERAR ≥50")
fig.update_layout(width=1000, height=max(400, len(df_ranking) * 35))
fig.show()


In [19]:
# ── 4.9 MC3 Absoluto × Sell-through 30d ──────────────────────────
# Visão simplificada sem scores — direto nos KPIs brutos.
# Acionável:
#   - Growth: produtos com MC3 alto mas sellthrough baixo → oportunidade de push
#   - Produto: MC3 negativo → rever precificação ou CMV

if "mc3" in df_scorecard.columns and "sellthrough_30d" in df_scorecard.columns:
    df_mc = df_scorecard.dropna(subset=["mc3", "sellthrough_30d"]).copy()

    fig = px.scatter(
        df_mc,
        x="sellthrough_30d",
        y="mc3",
        color="classificacao",
        color_discrete_map=COLOR_MAP_CLASSIFICACAO,
        hover_name="product_name",
        hover_data=["category_4", "receita_liquida"],
        title="MC3 × Sell-through 30d — Rentabilidade vs Tração",
        labels={
            "sellthrough_30d": "Sell-through 30d",
            "mc3": "MC3 (margem)",
        },
    )
    fig.add_hline(
        y=SCORECARD_INPUTS["benchmark_mc3"],
        line_dash="dash", line_color="blue", opacity=0.5,
        annotation_text=f"Benchmark MC3 ({SCORECARD_INPUTS['benchmark_mc3']:.0%})",
    )
    fig.update_layout(
        width=900, height=550,
        xaxis_tickformat=".0%",
        yaxis_tickformat=".0%",
    )
    fig.show()


In [20]:
# ── 4.10 Checklist: Taxa de Aprovação por Critério ────────────────
# Visão executiva: quais critérios do checklist são mais difíceis de atingir?
# Critérios com baixa aprovação indicam gaps sistêmicos nos lançamentos.

check_cols = [
    c for c in df_checklist.columns
    if c != "Produto" and not c.endswith(" value")
]
if check_cols:
    pass_rates = df_checklist[check_cols].mean().sort_values() * 100

    fig = px.bar(
        x=pass_rates.values,
        y=pass_rates.index,
        orientation="h",
        color=pass_rates.values,
        color_continuous_scale=["#e74c3c", "#f39c12", "#2ecc71"],
        title="Checklist: Taxa de Aprovação por Critério (%)",
        labels={"x": "% de Produtos Aprovados", "y": "Critério"},
    )
    fig.update_layout(width=900, height=max(400, len(check_cols) * 40))
    fig.show()


## 5. Exportação para Google Sheets

Duas planilhas são atualizadas:
1. **Scorecard** → abas "export long" (todas as colunas) e "export short" (resumo por produto)
2. **Checklist** → aba "output checklist" (com checkboxes e formatação condicional)


In [21]:
# ── Exportar Scorecard para Sheets ────────────────────────────────

# --- deepnote Long ---
ws_long = SPREADSHEET_SCORECARD.worksheet("deepnote long")
df_long = df_scorecard.copy()
df_long["data_lancamento"] = (
    pd.to_datetime(df_long["data_lancamento"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
)
df_long.sort_values(by="data_lancamento", inplace=True)

for col in df_long.columns:
    if pd.api.types.is_extension_array_dtype(df_long[col]):
        df_long[col] = df_long[col].astype(object)

ws_long.batch_clear(ranges=["A1:Z1000"])

df_long = df_long.fillna(" ")
df_long = df_long[["classificacao"] + df_long.columns[:-1].tolist()]
cols_long = [str(c).replace("__", " - ").replace("_", " ").lower() for c in df_long.columns]
ws_long.update(range_name="A1", values=[cols_long] + df_long.values.tolist())

# --- deepnote Short ---
ws_short = SPREADSHEET_SCORECARD.worksheet("deepnote short")

ws_short.batch_clear(ranges=["A1:I1000"])

df_short = df_scorecard[id_cols_preview + score_cols_preview].copy()
df_short["data_lancamento"] = (
    pd.to_datetime(df_short["data_lancamento"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
)
df_short.sort_values(by="data_lancamento", inplace=True)

for col in df_short.columns:
    if pd.api.types.is_extension_array_dtype(df_short[col]):
        df_short[col] = df_short[col].astype(object)

df_short = df_short.fillna(" ")
cols_short = [str(c).replace("_", " ").lower() for c in df_short.columns]
ws_short.update(range_name="A1", values=[cols_short] + df_short.values.tolist())


# --- Products Main Infos ---

main_info_cols = ['product_name', 'category_4',
        'cobertura_dias',
        'tempo_de_vendas', 'sellthrough_30d', 'sellthrough_60d',
        'sellthrough_90d', 'mrkup_entrada', 'mrkup_med', 'full_price', 'cmv_med', 
        'tx_devolucao', 'tx_devolucao_categoria','rating', 'mc3', 'mc3_categoria', 
        'score_pilar__tracao_comercial',
        'score_pilar__unit_economics',
        'score_pilar__satisfacao_e_marca',
        'score_total',
        'classificacao'
       ]

ws_main_infos = SPREADSHEET_SCORECARD.worksheet("deepnote main infos")

ws_main_infos.batch_clear(ranges=["A1:Z1000"])

df_main_infos = df_scorecard[main_info_cols].copy()

df_main_infos.sort_values(by="tempo_de_vendas", ascending=False, inplace=True)

for col in df_main_infos.columns:
    if pd.api.types.is_extension_array_dtype(df_main_infos[col]):
        df_main_infos[col] = df_main_infos[col].astype(object)

df_main_infos = df_main_infos.fillna(" ")
cols_main_infos = [str(c).replace("_", " ").lower() for c in df_main_infos.columns]
ws_main_infos.update(range_name="A1", values=[cols_main_infos] + df_main_infos.values.tolist())




print("✓ Scorecard exportado (deepnote long + deepnote short)")

✓ Scorecard exportado (deepnote long + deepnote short)


In [22]:
# ── Exportar Checklist para Sheets ────────────────────────────────

sheet = SPREADSHEET_CHECKLIST.worksheet("output checklist")

# Monta tabela com colunas alternadas: check → valor
ordered_columns = ["Produto"]
rename_map = {}
for col in df_checklist.columns:
    if col == "Produto" or col.endswith(" value"):
        continue
    ordered_columns.append(col)
    value_col = f"{col} value"
    if value_col in df_checklist.columns:
        ordered_columns.append(value_col)
        rename_map[value_col] = f"{col} - valor"

# Seleciona e renomeia
df_output = df_checklist[ordered_columns].rename(columns=rename_map).copy()

# Normaliza tipos antes de arredondar para evitar operações em colunas booleanas
for c in df_output.columns:
    if c != "Produto":
        # Se a coluna for booleana, mantém como bool (checkbox)
        if pd.api.types.is_bool_dtype(df_output[c]):
            # deixa como bool
            continue
        # Se for numérica (float/int), arredonda
        if pd.api.types.is_numeric_dtype(df_output[c]):
            df_output[c] = pd.to_numeric(df_output[c], errors="coerce").round(2)
        else:
            # Tenta converter strings numéricas das colunas " - valor" para números
            if str(c).endswith(" - valor"):
                df_output[c] = pd.to_numeric(df_output[c], errors="coerce").round(2)

# Colunas de checkbox (excluindo valores numéricos)
checkbox_cols = [
    c for c in df_output.columns
    if c != "Produto" and not c.endswith(" - valor")
]


def to_bool_safe(v):
    """Converte valores para booleano de forma segura."""
    if pd.isna(v):
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return v != 0
    if isinstance(v, str):
        return v.strip().lower() in {"true", "1", "sim", "yes", "y", "t", "verdadeiro"}
    return False


# Garante que colunas de checkbox sejam booleanas puras
for c in checkbox_cols:
    df_output[c] = df_output[c].apply(to_bool_safe).astype(bool)

# Coluna final com total de checks TRUE
# Usa soma sobre booleanos (True=1, False=0), depois converte para int
df_output["Qtd checks"] = df_output[checkbox_cols].sum(axis=1).astype(int)

# Headers de grupo (linha 1) e detalhados (linha 2)
headers = [" " if c == "Produto" else c.split(" - ")[0] for c in df_output.columns]
# Substitui NaN por string para envio ao Sheets
data_to_insert = [df_output.columns.tolist()] + df_output.fillna(" ").values.tolist()

# Limpa e atualiza o conteúdo
sheet.batch_clear(["A1:P1000"])
sheet.update("A1", [headers])
sheet.update("A2", data_to_insert)


def col_to_a1(col_idx):
    """Converte índice de coluna (0-based) em letra A1."""
    col_idx += 1
    letters = ""
    while col_idx > 0:
        col_idx, remainder = divmod(col_idx - 1, 26)
        letters = chr(65 + remainder) + letters
    return letters


# Aplica validação de checkbox + formatação condicional verde
requests = []
for col in checkbox_cols:
    col_idx = df_output.columns.get_loc(col)
    col_a1 = col_to_a1(col_idx)

    # Checkbox
    requests.append({
        "setDataValidation": {
            "range": {
                "sheetId": sheet.id,
                "startRowIndex": 2,
                "endRowIndex": 2 + len(df_output),
                "startColumnIndex": col_idx,
                "endColumnIndex": col_idx + 1,
            },
            "rule": {
                "condition": {"type": "BOOLEAN"},
                "strict": False,
                "showCustomUi": True,
            },
        }
    })

    # Verde quando TRUE
    requests.append({
        "addConditionalFormatRule": {
            "rule": {
                "ranges": [{
                    "sheetId": sheet.id,
                    "startRowIndex": 2,
                    "endRowIndex": 2 + len(df_output),
                    "startColumnIndex": col_idx,
                    "endColumnIndex": col_idx + 1,
                }],
                "booleanRule": {
                    "condition": {
                        "type": "CUSTOM_FORMULA",
                        "values": [{"userEnteredValue": f"={col_a1}3=TRUE"}],
                    },
                    "format": {
                        "backgroundColor": {
                            "red": 0.85, "green": 0.95, "blue": 0.85,
                        },
                        "textFormat": {
                            "foregroundColor": {
                                "red": 0.22, "green": 0.47, "blue": 0.22,
                            },
                            "bold": True,
                        },
                    },
                },
            },
            "index": 0,
        }
    })

if requests:
    SPREADSHEET_CHECKLIST.batch_update({"requests": requests})

print("✓ Checklist exportado com checkboxes e formatação condicional")

/tmp/ipykernel_116/1950610469.py:70: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update("A1", [headers])
/tmp/ipykernel_116/1950610469.py:71: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update("A2", data_to_insert)
✓ Checklist exportado com checkboxes e formatação condicional


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1d874582-dea2-475f-ab7f-876ea0abc9ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>